# Baseline RAG System for Cultural Events

This notebook implements a baseline RAG (Retrieval-Augmented Generation) system for cultural events in Savoie (73), Haute-Savoie (74), and Isère (38).

## Architecture
- **Embeddings**: Mistral AI embeddings
- **Vector Store**: FAISS (CPU)
- **LLM**: Mistral AI (mistral-small-latest)
- **Framework**: LangChain

## 1. Setup & Imports

In [1]:
import json
import os
from pathlib import Path
from datetime import datetime

import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_mistralai import MistralAIEmbeddings, ChatMistralAI
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

# Load environment variables
load_dotenv()

# Verify Mistral API key
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
if not MISTRAL_API_KEY:
    raise ValueError("MISTRAL_API_KEY not found in environment. Please check your .env file.")

print("✓ All imports successful")
print(f"✓ Mistral API key loaded (length: {len(MISTRAL_API_KEY)} characters)")

✓ All imports successful
✓ Mistral API key loaded (length: 32 characters)


## 2. Load and Explore Data

In [2]:
# Load filtered JSON data
DATA_PATH = Path("../data/processed/events_filtered.json")

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    events = json.load(f)

print(f"Total events loaded: {len(events):,}")

# Convert to DataFrame for exploration
df = pd.DataFrame(events)

# Display statistics
print("\n" + "="*60)
print("DATASET STATISTICS")
print("="*60)

# Department distribution
print("\nDepartment Distribution:")
print(df['location_department'].value_counts())

# Date range
df['firstdate_begin_dt'] = pd.to_datetime(df['firstdate_begin'], errors='coerce')
print(f"\nDate Range:")
print(f"  Earliest: {df['firstdate_begin_dt'].min()}")
print(f"  Latest: {df['firstdate_begin_dt'].max()}")

# Category distribution (top 10)
print("\nTop 10 Categories:")
print(df['category'].value_counts().head(10))

# Sample event preview
print("\n" + "="*60)
print("SAMPLE EVENT")
print("="*60)
sample = events[0]
print(f"Title: {sample.get('title_fr', 'N/A')}")
print(f"City: {sample.get('location_city', 'N/A')}")
print(f"Department: {sample.get('location_department', 'N/A')}")
print(f"Date: {sample.get('daterange_fr', 'N/A')}")
print(f"Category: {sample.get('category', 'N/A')}")
print(f"\nDescription: {sample.get('description_fr', 'N/A')[:200]}...")

Total events loaded: 8,542

DATASET STATISTICS

Department Distribution:
location_department
Isère           3482
Haute-Savoie    2719
Savoie          2341
Name: count, dtype: int64

Date Range:
  Earliest: 2023-01-02 09:00:00+01:00
  Latest: 2032-01-01 02:00:00+01:00

Top 10 Categories:
Series([], Name: count, dtype: int64)

SAMPLE EVENT
Title: Recrutement d'ouvriers forestiers H/F pour l'ONF 🌲🌳🌲
City: Moûtiers
Department: Savoie
Date: Jeudi 4 avril, 09h00
Category: None

Description: Venez rencontrer les recruteurs de l'Office Nationale des Forêts pour travailler en saison du 02/05/24 au 30/10/2024....


/var/folders/lc/mfqjb84972q1sr1b4rxzjnhm0000gn/T/ipykernel_60816/3648425553.py:22: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['firstdate_begin_dt'] = pd.to_datetime(df['firstdate_begin'], errors='coerce')


## 3. Data Preprocessing

In [5]:
def clean_html(html_text):
    """Remove HTML tags from text."""
    if not html_text:
        return ""
    soup = BeautifulSoup(html_text, 'lxml')
    return soup.get_text(separator=' ', strip=True)


def build_document_text(event):
    """
    Build comprehensive document text from event data.
    Combines title, description, location, and date information.
    """
    parts = []
    
    # Title
    title = event.get('title_fr', '')
    if title:
        parts.append(f"Événement: {title}")
    
    # Description (prefer long description, fallback to short)
    long_desc = clean_html(event.get('longdescription_fr', ''))
    short_desc = event.get('description_fr', '')
    description = long_desc if long_desc else short_desc
    if description:
        parts.append(f"Description: {description}")
    
    # Location
    city = event.get('location_city', '')
    location_name = event.get('location_name', '')
    address = event.get('location_address', '')
    department = event.get('location_department', '')
    
    location_parts = []
    if location_name:
        location_parts.append(location_name)
    if address:
        location_parts.append(address)
    if city:
        location_parts.append(city)
    if department:
        location_parts.append(department)
    
    if location_parts:
        parts.append(f"Lieu: {', '.join(location_parts)}")
    
    # Date
    daterange = event.get('daterange_fr', '')
    if daterange:
        parts.append(f"Date: {daterange}")
    
    # Category
    category = event.get('category', '')
    if category:
        parts.append(f"Catégorie: {category}")
    
    # Keywords
    keywords = event.get('keywords_fr', '')
    if keywords:
        parts.append(f"Mots-clés: {keywords}")
    
    return '\n'.join(parts)


# Create Document objects with metadata
documents = []

for event in events:
    doc_text = build_document_text(event)
    
    # Create metadata
    metadata = {
        'title': event.get('title_fr', ''),
        'city': event.get('location_city', ''),
        'department': event.get('location_department', ''),
        'category': event.get('category', ''),
        'daterange': event.get('daterange_fr', ''),
        'firstdate': event.get('firstdate_begin', ''),
    }
    
    documents.append(Document(page_content=doc_text, metadata=metadata))

print(f"✓ Created {len(documents):,} documents")
print("\nSample document:")
print("="*60)
print(documents[0].page_content[:500])
print("...")
print("\nMetadata:", documents[0].metadata)

✓ Created 8,542 documents

Sample document:
Événement: Recrutement d'ouvriers forestiers H/F pour l'ONF 🌲🌳🌲
Description: Venez rencontrer les recruteurs de l'Office Nationale des Forêts pour travailler en saison du 02/05/24 au 30/10/2024. Venir avec un CV à jour Présentation de l'entreprise, des postes et de leurs conditions Entretiens d'embauche
Lieu: Moûtiers, 73600 Moûtiers, Moûtiers, Savoie
Date: Jeudi 4 avril, 09h00
Mots-clés: ['Job dating', 'Recrutement', 'Tout public', 'Contrat saisonnier', 'CDD', 'en physique']
...

Metadata: {'title': "Recrutement d'ouvriers forestiers H/F pour l'ONF 🌲🌳🌲", 'city': 'Moûtiers', 'department': 'Savoie', 'category': None, 'daterange': 'Jeudi 4 avril, 09h00', 'firstdate': '2024-04-04T09:00:00+02:00'}


## 4. Embedding Setup

In [7]:
# Initialize Mistral embeddings
embeddings = MistralAIEmbeddings(
    model="mistral-embed",
    api_key=MISTRAL_API_KEY
)

# Test with sample text
test_text = "Concert de musique classique à Annecy"
test_embedding = embeddings.embed_query(test_text)

print(f"✓ Embeddings initialized")
print(f"✓ Test embedding generated")
print(f"  - Embedding dimension: {len(test_embedding)}")
print(f"  - First 5 values: {test_embedding[:5]}")

tokenizer.json: 0.00B [00:00, ?B/s]

✓ Embeddings initialized
✓ Test embedding generated
  - Embedding dimension: 1024
  - First 5 values: [0.004291534423828125, 0.0194854736328125, 0.07293701171875, 0.019317626953125, 0.029144287109375]


## 5. FAISS Index Creation

In [9]:
print("Creating FAISS index... This may take a few minutes.")
print(f"Processing {len(documents):,} documents...")

# Create FAISS index from documents
vectorstore = FAISS.from_documents(documents, embeddings)

# Save index to disk
INDEX_PATH = "../data/index/faiss_baseline"
Path(INDEX_PATH).parent.mkdir(parents=True, exist_ok=True)
vectorstore.save_local(INDEX_PATH)

print(f"\n✓ FAISS index created successfully")
print(f"✓ Index saved to: {INDEX_PATH}")
print(f"  - Number of documents indexed: {len(documents):,}")

# Test retrieval
test_query = "concerts à Annecy"
test_results = vectorstore.similarity_search(test_query, k=3)

print(f"\nTest retrieval for query: '{test_query}'")
print(f"Retrieved {len(test_results)} documents")
print("\nTop result:")
print("="*60)
print(test_results[0].page_content[:300])
print("...")
print(f"Metadata: {test_results[0].metadata}")

Creating FAISS index... This may take a few minutes.
Processing 8,542 documents...

✓ FAISS index created successfully
✓ Index saved to: ../data/index/faiss_baseline
  - Number of documents indexed: 8,542

Test retrieval for query: 'concerts à Annecy'
Retrieved 3 documents

Top result:
Événement: Concerts au château
Description: Musique et danse au Musée-Château d’Annecy ! Le Conservatoire à Rayonnement Régional fera vibrer le château le temps d’une soirée, en lien avec le Projet Scientifique et Culturel du musée autour du mouvement et de ses représentations à travers l’histoire d
...
Metadata: {'title': 'Concerts au château', 'city': 'Annecy', 'department': 'Haute-Savoie', 'category': None, 'daterange': 'Samedi 13 mai, 18h00', 'firstdate': '2023-05-13T18:00:00+02:00'}


## 6. RAG Chain Setup

In [10]:
# Initialize Mistral LLM
llm = ChatMistralAI(
    model="mistral-small-latest",
    api_key=MISTRAL_API_KEY,
    temperature=0.1
)

# Create custom prompt template
prompt_template = """Tu es un assistant spécialisé dans les événements culturels de Savoie, Haute-Savoie et Isère.
Utilise les informations suivantes pour répondre à la question de l'utilisateur.
Si tu ne trouves pas l'information dans le contexte, dis-le clairement.

Contexte:
{context}

Question: {question}

Réponse détaillée:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# Build RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

print("✓ RAG chain initialized successfully")
print(f"  - LLM: mistral-small-latest")
print(f"  - Retriever: similarity search (k=5)")
print(f"  - Chain type: stuff")

✓ RAG chain initialized successfully
  - LLM: mistral-small-latest
  - Retriever: similarity search (k=5)
  - Chain type: stuff


## 7. Testing & Validation

In [11]:
# Test queries
test_queries = [
    "Quels concerts ont lieu à Annecy?",
    "Événements pour enfants à Grenoble",
    "Festivals de musique en Savoie"
]

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*60}")
    print(f"TEST QUERY {i}: {query}")
    print(f"{'='*60}")
    
    result = qa_chain.invoke({"query": query})
    
    # Display answer
    print("\nRÉPONSE:")
    print(result['result'])
    
    # Display source documents
    print(f"\nSOURCES ({len(result['source_documents'])} documents):")
    for j, doc in enumerate(result['source_documents'], 1):
        print(f"\n  {j}. {doc.metadata.get('title', 'N/A')}")
        print(f"     Lieu: {doc.metadata.get('city', 'N/A')}, {doc.metadata.get('department', 'N/A')}")
        print(f"     Date: {doc.metadata.get('daterange', 'N/A')}")
        print(f"     Catégorie: {doc.metadata.get('category', 'N/A')}")


TEST QUERY 1: Quels concerts ont lieu à Annecy?

RÉPONSE:
À Annecy, plusieurs concerts sont programmés dans différents lieux. Voici les événements culturels musicaux mentionnés dans le contexte :

1. **Concerts au château**
   - **Lieu** : Musée-Château d’Annecy (Place du château, 74000 Annecy)
   - **Date et heure** : Samedi 13 mai, 18h00
   - **Description** : Musique et danse proposées par le Conservatoire à Rayonnement Régional, en lien avec le Projet Scientifique et Culturel du musée autour du mouvement et de ses représentations à travers l’histoire de l’art.

2. **Événements musicaux au Conservatoire d'art et d'histoire**
   - **Lieu** : Conservatoire d'art et d'histoire (18, avenue de Trésum, 74000 Annecy)
   - **Dates et horaires** :
     - **Samedi 20 septembre** : Répétitions de l’œuvre *"La messa di gloria"* de Puccini par la Chorale du Grand Ensemble Vocal d'Annecy (14h-17h).
     - **Dimanche 21 septembre** : Concerts de l'Orchestre des Pays de Savoie (quatuor à vents) à 

## 8. Basic Evaluation

In [16]:
# Manual evaluation with sample questions
evaluation_queries = [
    "Y a-t-il des expositions d'art à Chambéry?",
    "Quels sont les événements gratuits en Haute-Savoie?",
    "Événements en mars 2024 à Grenoble",
    "Spectacles de théâtre en Isère",
    "Ateliers créatifs pour enfants en Savoie"
]

print("MANUAL EVALUATION")
print("="*60)
print("Review the context and answers below to assess quality.\n")

for i, query in enumerate(evaluation_queries, 1):
    print(f"\n{'='*60}")
    print(f"EVALUATION {i}/5")
    print(f"{'='*60}")
    print(f"Question: {query}")
    print("-" * 60)
    
    result = qa_chain.invoke({"query": query})
    
    # Context
    print("\nCONTEXT (Retrieved Documents):")
    for j, doc in enumerate(result['source_documents'], 1):
        print(f"\n  [{j}] {doc.metadata.get('title', 'N/A')}")
        print(f"      {doc.metadata.get('city', 'N/A')} - {doc.metadata.get('daterange', 'N/A')}")
    
    # Answer
    print("\n" + "-" * 60)
    print("ANSWER:")
    print(result['result'])
    print("\n" + "=" * 60)

print("\n\nEvaluation complete. Review the responses above for:")
print("  1. Relevance: Are retrieved documents relevant to the query?")
print("  2. Accuracy: Does the answer match the context?")
print("  3. Completeness: Does the answer address the full question?")
print("  4. Coherence: Is the answer well-structured and clear?")

MANUAL EVALUATION
Review the context and answers below to assess quality.


EVALUATION 1/5
Question: Y a-t-il des expositions d'art à Chambéry?
------------------------------------------------------------

CONTEXT (Retrieved Documents):

  [1] A la découverte de l'artothèque
      Chambéry - Samedi 18 mai, 19h00

  [2] A la découverte de l'artothèque
      Chambéry - Samedi 13 mai, 20h00

  [3] Visite flash : Dans les pas des artistes…
      Chambéry - Samedi 18 mai, 22h00, 22h30, 23h00

  [4] L'Adresse au paysage. Figures de la montagne.
      Chambéry - Samedi 13 mai, 20h00

  [5] Visite flash : Arto Quezako ?
      Chambéry - 20 et 21 septembre

------------------------------------------------------------
ANSWER:
Oui, il y a des expositions d'art à Chambéry, notamment au **Musée des Beaux-Arts de Chambéry**. Voici les expositions mentionnées dans le contexte :

1. **Exposition temporaire : *Rêveries de promeneurs solitaires***
   - **Thème** : La marche et ses liens avec des œuvres 

## 9. Hybrid RAG (Dense + Sparse Retrieval)

In [18]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

print("Creating Hybrid RAG system...")

# Create BM25 retriever from documents
print("  - Initializing BM25 retriever...")
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 5  # Number of documents to retrieve

# Get FAISS retriever
print("  - Setting up FAISS retriever...")
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Create ensemble retriever with Reciprocal Rank Fusion
print("  - Creating ensemble retriever (RRF)...")
ensemble_retriever = EnsembleRetriever(
    retrievers=[faiss_retriever, bm25_retriever],
    weights=[0.5, 0.5]  # Equal weights for dense and sparse
)

# Build hybrid RAG chain
print("  - Building Hybrid RAG chain...")
hybrid_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=ensemble_retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

print("\n✓ Hybrid RAG system initialized successfully")
print(f"  - Dense retriever: FAISS (semantic similarity)")
print(f"  - Sparse retriever: BM25 (keyword matching)")
print(f"  - Fusion method: Reciprocal Rank Fusion")
print(f"  - Weights: 0.5 dense + 0.5 sparse")

# Test comparison between Basic RAG and Hybrid RAG
test_query = "festival jazz Chambéry juillet 2024"

print(f"\n{'='*60}")
print(f"COMPARISON TEST: Basic RAG vs Hybrid RAG")
print(f"{'='*60}")
print(f"Query: '{test_query}'")
print(f"{'='*60}")

# Basic RAG result
print("\n[BASIC RAG - Dense only]")
basic_result = qa_chain.invoke({"query": test_query})
print("\nRéponse:")
print(basic_result['result'])
print(f"\nSources ({len(basic_result['source_documents'])} docs):")
for i, doc in enumerate(basic_result['source_documents'][:3], 1):
    print(f"  {i}. {doc.metadata['title'][:60]}... ({doc.metadata['city']})")

# Hybrid RAG result
print(f"\n{'-'*60}")
print("\n[HYBRID RAG - Dense + Sparse]")
hybrid_result = hybrid_qa_chain.invoke({"query": test_query})
print("\nRéponse:")
print(hybrid_result['result'])
print(f"\nSources ({len(hybrid_result['source_documents'])} docs):")
for i, doc in enumerate(hybrid_result['source_documents'][:3], 1):
    print(f"  {i}. {doc.metadata['title'][:60]}... ({doc.metadata['city']})")

print(f"\n{'='*60}")
print("✓ Hybrid RAG comparison complete")

Creating Hybrid RAG system...
  - Initializing BM25 retriever...
  - Setting up FAISS retriever...
  - Creating ensemble retriever (RRF)...
  - Building Hybrid RAG chain...

✓ Hybrid RAG system initialized successfully
  - Dense retriever: FAISS (semantic similarity)
  - Sparse retriever: BM25 (keyword matching)
  - Fusion method: Reciprocal Rank Fusion
  - Weights: 0.5 dense + 0.5 sparse

COMPARISON TEST: Basic RAG vs Hybrid RAG
Query: 'festival jazz Chambéry juillet 2024'

[BASIC RAG - Dense only]

Réponse:
D'après les informations disponibles dans le contexte fourni, il n'y a pas de mention d'un festival de jazz à Chambéry en juillet 2024.

Cependant, voici quelques événements musicaux à Chambéry ou dans les environs proches (Savoie, Haute-Savoie, Isère) qui pourraient vous intéresser :

1. **Festival de rue en cour(s)!** (Chambéry, 24-30 juillet 2024) : Bien que ce festival ne soit pas spécifiquement dédié au jazz, il propose des spectacles de rue variés, dont certains pourraient i

## 10. Advanced RAG (Query Analysis + HyDE + Reranking)

In [20]:
from langchain_community.document_compressors import FlashrankRerank
from langchain_classic.retrievers import ContextualCompressionRetriever

print("Creating Advanced RAG system...")
print("="*60)

# =============================================================================
# 1. QUERY ANALYSIS FUNCTION
# =============================================================================

def analyze_query(query: str, llm_instance) -> dict:
    """
    Analyze query to determine:
    1. Is it relevant to cultural events in Savoie/Haute-Savoie/Isère?
    2. Is it a valid question (not offensive, not gibberish)?
    3. Should we proceed with retrieval?
    4. Reformulated query for optimal retrieval
    """
    import json as json_module
    
    analysis_prompt_template = """Tu es un assistant spécialisé dans les événements culturels de Savoie, Haute-Savoie et Isère.

Analyse la question suivante et réponds en JSON:

Question: {query}

Structure de notre base de données:
- Événements culturels (concerts, festivals, expositions, théâtre, etc.)
- Localisation: villes en Savoie, Haute-Savoie, Isère
- Dates: événements de 2023 à aujourd'hui
- Catégories: musique, art, théâtre, enfants, etc.

Réponds UNIQUEMENT avec ce JSON (pas d'autre texte):
{{
    "is_relevant": true/false,
    "reason": "explication courte si non pertinent",
    "should_search": true/false,
    "reformulated_query": "version optimisée de la question pour la recherche",
    "search_focus": "ce qu'il faut chercher (lieu, date, type d'événement, etc.)"
}}"""
    
    analysis_prompt = PromptTemplate(
        template=analysis_prompt_template,
        input_variables=["query"]
    )
    
    # Get LLM response
    response = llm_instance.invoke(analysis_prompt.format(query=query))
    response_text = response.content if hasattr(response, 'content') else str(response)
    
    # Parse JSON response
    try:
        # Extract JSON from markdown code blocks if present
        if "```json" in response_text:
            response_text = response_text.split("```json")[1].split("```")[0].strip()
        elif "```" in response_text:
            response_text = response_text.split("```")[1].split("```")[0].strip()
        
        analysis = json_module.loads(response_text)
        return analysis
    except Exception as e:
        # Fallback: assume query is relevant if parsing fails
        return {
            "is_relevant": True,
            "reason": "",
            "should_search": True,
            "reformulated_query": query,
            "search_focus": "événements culturels"
        }

print("✓ Query analysis function defined")

# =============================================================================
# 2. HYPOTHETICAL DOCUMENT GENERATION (HyDE)
# =============================================================================

def generate_hypothetical(query: str, llm_instance) -> str:
    """
    Generate a hypothetical event description that would answer the query.
    This improves retrieval by creating a document-like representation of the answer.
    """
    hyde_prompt_template = """Imagine un événement culturel qui répondrait parfaitement à cette question.

Question: {query}

Décris cet événement hypothétique de manière détaillée (titre, lieu, date approximative, description):"""
    
    hyde_prompt = PromptTemplate(
        template=hyde_prompt_template,
        input_variables=["query"]
    )
    
    response = llm_instance.invoke(hyde_prompt.format(query=query))
    return response.content if hasattr(response, 'content') else str(response)

print("✓ HyDE generation function defined")

# =============================================================================
# 3. RERANKER SETUP
# =============================================================================

print("  - Initializing FlashRank reranker...")
reranker = FlashrankRerank(top_n=5, model="ms-marco-MiniLM-L-12-v2")

print("✓ Reranker initialized")

# =============================================================================
# 4. ADVANCED RAG FUNCTION
# =============================================================================

def advanced_rag(query: str) -> dict:
    """
    Advanced RAG with query analysis, HyDE, and reranking.
    
    Flow:
    1. Analyze query (is it relevant?)
    2. If not relevant, return direct response
    3. If relevant, generate hypothetical document (HyDE)
    4. Search with HyDE embedding
    5. Rerank results
    6. Generate final answer
    """
    print(f"\n{'='*60}")
    print(f"ADVANCED RAG PROCESSING")
    print(f"{'='*60}")
    print(f"Query: {query}")
    
    # Step 1: Query Analysis
    print("\n[1/5] Analyzing query...")
    analysis = analyze_query(query, llm)
    print(f"  - Relevant: {analysis['is_relevant']}")
    print(f"  - Should search: {analysis['should_search']}")
    
    if not analysis["should_search"]:
        print(f"  - Bypassing search: {analysis['reason']}")
        return {
            "answer": analysis["reason"],
            "source_documents": [],
            "query_analysis": analysis,
            "bypassed_search": True,
            "hypothetical_doc": None
        }
    
    # Step 2: HyDE - Generate hypothetical answer
    print(f"\n[2/5] Generating hypothetical document (HyDE)...")
    reformulated = analysis["reformulated_query"]
    print(f"  - Reformulated query: {reformulated}")
    hypothetical_answer = generate_hypothetical(reformulated, llm)
    print(f"  - Hypothetical doc (first 100 chars): {hypothetical_answer[:100]}...")
    
    # Step 3: Search with HyDE embedding
    print(f"\n[3/5] Searching with HyDE embedding...")
    hyde_embedding = embeddings.embed_query(hypothetical_answer)
    initial_docs = vectorstore.similarity_search_by_vector(hyde_embedding, k=10)
    print(f"  - Retrieved {len(initial_docs)} documents")
    
    # Step 4: Rerank results
    print(f"\n[4/5] Reranking documents...")
    reranked_docs = reranker.compress_documents(initial_docs, reformulated)[:5]
    print(f"  - Top {len(reranked_docs)} documents after reranking")
    
    # Step 5: Generate final answer
    print(f"\n[5/5] Generating final answer...")
    context = "\n\n".join([doc.page_content for doc in reranked_docs])
    final_prompt = PROMPT.format(context=context, question=query)
    final_answer = llm.invoke(final_prompt)
    
    print(f"✓ Advanced RAG processing complete")
    
    return {
        "answer": final_answer.content if hasattr(final_answer, 'content') else str(final_answer),
        "source_documents": reranked_docs,
        "query_analysis": analysis,
        "hypothetical_doc": hypothetical_answer,
        "bypassed_search": False
    }

print("✓ Advanced RAG function defined")

print(f"\n{'='*60}")
print("✓ Advanced RAG system initialized successfully")
print(f"{'='*60}")
print("Components:")
print("  - Query Analysis: LLM-based relevance check")
print("  - HyDE: Hypothetical document generation")
print("  - Reranking: FlashRank (ms-marco-MiniLM-L-12-v2)")
print("  - Retrieval: Top 10 → Rerank to Top 5")
print(f"{'='*60}")

INFO:flashrank.Ranker:Downloading ms-marco-MiniLM-L-12-v2...


Creating Advanced RAG system...
✓ Query analysis function defined
✓ HyDE generation function defined
  - Initializing FlashRank reranker...


ms-marco-MiniLM-L-12-v2.zip: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21.6M/21.6M [00:00<00:00, 72.1MiB/s]


✓ Reranker initialized
✓ Advanced RAG function defined

✓ Advanced RAG system initialized successfully
Components:
  - Query Analysis: LLM-based relevance check
  - HyDE: Hypothetical document generation
  - Reranking: FlashRank (ms-marco-MiniLM-L-12-v2)
  - Retrieval: Top 10 → Rerank to Top 5


In [22]:
# Test Advanced RAG with different query types

test_queries_advanced = [
    # Standard query - should work well
    ("Activités culturelles pour enfants près de Grenoble", "standard"),
    
    # Off-topic query - should bypass search
    ("Comment faire une tarte aux pommes?", "off-topic"),
    
    # Complex query - should benefit from HyDE and reranking
    ("Je cherche des événements culturels gratuits adaptés aux familles avec enfants de moins de 10 ans, idéalement en week-end et accessibles depuis Annecy", "complex"),
]

for i, (query, query_type) in enumerate(test_queries_advanced, 1):
    print(f"\n{'#'*60}")
    print(f"TEST {i}/3: {query_type.upper()} QUERY")
    print(f"{'#'*60}")
    
    result = advanced_rag(query)
    
    print(f"\n{'='*60}")
    print("RESULTS:")
    print(f"{'='*60}")
    
    # Query analysis
    print("\n[Query Analysis]")
    print(f"  Relevant: {result['query_analysis']['is_relevant']}")
    print(f"  Should search: {result['query_analysis']['should_search']}")
    print(f"  Reformulated: {result['query_analysis']['reformulated_query']}")
    
    if result['bypassed_search']:
        print(f"\n[Search Bypassed]")
        print(f"  Reason: {result['query_analysis']['reason']}")
    else:
        print(f"\n[HyDE Document]")
        print(f"  {result['hypothetical_doc'][:200]}...")
        
        print(f"\n[Retrieved Sources] ({len(result['source_documents'])} docs)")
        for j, doc in enumerate(result['source_documents'], 1):
            print(f"  {j}. {doc.metadata['title'][:50]}...")
            print(f"     {doc.metadata['city']}, {doc.metadata['department']} - {doc.metadata['daterange']}")
    
    print(f"\n[Final Answer]")
    print(result['answer'])
    print(f"\n{'#'*60}\n")


############################################################
TEST 1/3: STANDARD QUERY
############################################################

ADVANCED RAG PROCESSING
Query: Activités culturelles pour enfants près de Grenoble

[1/5] Analyzing query...


INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"


  - Relevant: True
  - Should search: True

[2/5] Generating hypothetical document (HyDE)...
  - Reformulated query: activités culturelles pour enfants à Grenoble en Isère


INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"


  - Hypothetical doc (first 100 chars): **Événement culturel pour enfants à Grenoble en Isère :**
**"Les Petits Explorateurs de la Culture"*...

[3/5] Searching with HyDE embedding...


...

  - Hypothetical doc (first 100 chars): **Événement culturel : "Les Petits Explorateurs d'Annecy"**

**Lieu :** Parc Charles Bosson (Annecy,...

[3/5] Searching with HyDE embedding...


INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"


  - Retrieved 10 documents

[4/5] Reranking documents...
  - Top 5 documents after reranking

[5/5] Generating final answer...


INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"


✓ Advanced RAG processing complete

RESULTS:

[Query Analysis]
  Relevant: True
  Should search: True
  Reformulated: Événements culturels gratuits pour familles avec enfants de moins de 10 ans en week-end accessibles depuis Annecy en Savoie, Haute-Savoie ou Isère

[HyDE Document]
  **Événement culturel : "Les Petits Explorateurs d'Annecy"**

**Lieu :** Parc Charles Bosson (Annecy, Haute-Savoie) – Espace vert central, accessible en voiture, vélo ou transports en commun (bus ligne...

[Retrieved Sources] (5 docs)
  1. A la découverte du milieu montagnard 'La Haute-Sav...
     Seytroux, Haute-Savoie - 31 juillet - 11 août 2023
  2. Explorateur des Montagnes - Croq' Vacances...
     Châtel, Haute-Savoie - 21 - 27 juillet
  3. Explorateur des Montagnes - Croq' Vacances...
     Châtel, Haute-Savoie - 18 - 24 août
  4. Explorateur des Montagnes - Croq' Vacances...
     Châtel, Haute-Savoie - 14 - 20 juillet
  5. Explorateur des Montagnes - Croq' Vacances...
     Châtel, Haute-Savoie - 11 - 1

## 11. Comprehensive Comparison: Three RAG Flavors

In [24]:
# Comprehensive comparison of all three RAG approaches

comparison_query = "Événements musicaux gratuits pour familles à Annecy ce week-end"

print("="*80)
print("COMPREHENSIVE COMPARISON: THREE RAG FLAVORS")
print("="*80)
print(f"\nQuery: '{comparison_query}'")
print("="*80)

# ==================== BASIC RAG ====================
print("\n" + "▼"*80)
print("1. BASIC RAG (Dense Vector Search Only)")
print("▼"*80)
print("\nMethod: FAISS similarity search (top-5)")
print("Strengths: Fast, simple, good for semantic matching")
print("Weaknesses: May miss exact keyword matches\n")

basic_start = datetime.now()
basic_result = qa_chain.invoke({"query": comparison_query})
basic_time = (datetime.now() - basic_start).total_seconds()

print(f"[Execution time: {basic_time:.2f}s]")
print(f"\nRetrieved Sources ({len(basic_result['source_documents'])} docs):")
for i, doc in enumerate(basic_result['source_documents'][:3], 1):
    print(f"  {i}. {doc.metadata['title'][:60]}")
    print(f"     {doc.metadata['city']}, {doc.metadata['daterange']}")

print(f"\nAnswer:")
print(basic_result['result'][:300] + "...")

# ==================== HYBRID RAG ====================
print("\n\n" + "▼"*80)
print("2. HYBRID RAG (Dense + Sparse with RRF)")
print("▼"*80)
print("\nMethod: FAISS (semantic) + BM25 (keywords) with Reciprocal Rank Fusion")
print("Strengths: Combines semantic and keyword matching")
print("Weaknesses: More complex, may retrieve more diverse results\n")

hybrid_start = datetime.now()
hybrid_result = hybrid_qa_chain.invoke({"query": comparison_query})
hybrid_time = (datetime.now() - hybrid_start).total_seconds()

print(f"[Execution time: {hybrid_time:.2f}s]")
print(f"\nRetrieved Sources ({len(hybrid_result['source_documents'])} docs):")
for i, doc in enumerate(hybrid_result['source_documents'][:3], 1):
    print(f"  {i}. {doc.metadata['title'][:60]}")
    print(f"     {doc.metadata['city']}, {doc.metadata['daterange']}")

print(f"\nAnswer:")
print(hybrid_result['result'][:300] + "...")

# ==================== ADVANCED RAG ====================
print("\n\n" + "▼"*80)
print("3. ADVANCED RAG (Query Analysis + HyDE + Reranking)")
print("▼"*80)
print("\nMethod: LLM query analysis → HyDE → FAISS → FlashRank reranking")
print("Strengths: Most sophisticated, handles complex queries, filters irrelevant queries")
print("Weaknesses: Slowest, most LLM calls\n")

advanced_start = datetime.now()
advanced_result = advanced_rag(comparison_query)
advanced_time = (datetime.now() - advanced_start).total_seconds()

print(f"\n[Execution time: {advanced_time:.2f}s]")

if not advanced_result['bypassed_search']:
    print(f"\nQuery Analysis:")
    print(f"  Reformulated: {advanced_result['query_analysis']['reformulated_query']}")
    print(f"\nRetrieved Sources ({len(advanced_result['source_documents'])} docs):")
    for i, doc in enumerate(advanced_result['source_documents'][:3], 1):
        print(f"  {i}. {doc.metadata['title'][:60]}")
        print(f"     {doc.metadata['city']}, {doc.metadata['daterange']}")

print(f"\nAnswer:")
print(advanced_result['answer'][:300] + "...")

# ==================== SUMMARY ====================
print("\n\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"\n{'Approach':<20} {'Time (s)':<12} {'Retrieval Method':<40}")
print("-"*80)
print(f"{'Basic RAG':<20} {basic_time:<12.2f} {'FAISS (dense vectors)':<40}")
print(f"{'Hybrid RAG':<20} {hybrid_time:<12.2f} {'FAISS + BM25 (RRF fusion)':<40}")
print(f"{'Advanced RAG':<20} {advanced_time:<12.2f} {'Query Analysis + HyDE + Reranking':<40}")

print(f"\n{'Approach':<20} {'LLM Calls':<12} {'Best For':<50}")
print("-"*80)
print(f"{'Basic RAG':<20} {'1':<12} {'Simple queries, speed':<50}")
print(f"{'Hybrid RAG':<20} {'1':<12} {'Keyword-heavy queries, broader coverage':<50}")
print(f"{'Advanced RAG':<20} {'3+':<12} {'Complex queries, quality over speed':<50}")

print("\n" + "="*80)
print("✓ Comparison complete - All three RAG systems demonstrated")
print("="*80)

COMPREHENSIVE COMPARISON: THREE RAG FLAVORS

Query: 'Événements musicaux gratuits pour familles à Annecy ce week-end'

▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼
1. BASIC RAG (Dense Vector Search Only)
▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼

Method: FAISS similarity search (top-5)
Strengths: Fast, simple, good for semantic matching
Weaknesses: May miss exact keyword matches



INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"


[Execution time: 3.44s]

Retrieved Sources (5 docs):
  1. Visite famille art contemporain
     Annecy, 16 et 17 septembre
  2. The American Week-end
     Thonon-les-Bains, 10 et 11 juin
  3. Concerts au château
     Annecy, Samedi 13 mai, 18h00

Answer:
D'après les informations disponibles, voici les événements musicaux gratuits pour familles à Annecy ce week-end (16 et 17 septembre) :

1. **Visite famille art contemporain** (le dimanche 17 septembre à 14h30)
   - **Lieu** : Musée-château d'Annecy (1 place du Château, 74000 Annecy)
   - **Descripti...


▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼
2. HYBRID RAG (Dense + Sparse with RRF)
▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼

Method: FAISS (semantic) + BM25 (keywords) with Reciprocal Rank Fusion
Strengths: Combines semantic and keyword matching
Weaknesses: More complex, may retrieve more diverse results



INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"


[Execution time: 6.09s]

Retrieved Sources (10 docs):
  1. Visite famille art contemporain
     Annecy, 16 et 17 septembre
  2. ELIOR vous propose des postes de jour, de nuit, et même pour
     Pontcharra, Vendredi 16 février, 14h00
  3. The American Week-end
     Thonon-les-Bains, 10 et 11 juin

Answer:
D'après les informations disponibles dans le contexte, voici les événements musicaux gratuits pour familles à Annecy ce week-end (16 et 17 septembre) :

1. **Visite famille art contemporain** (Dimanche 17 septembre à 14h30)
   - **Lieu** : Musée-château d'Annecy, 1 place du Château, 74000 Annecy
   ...


▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼
3. ADVANCED RAG (Query Analysis + HyDE + Reranking)
▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼

Method: LLM query analysis → HyDE → FAISS → FlashRank reranking
Strengths: Most sophisticated, handles complex queries, filters irrelevant queries
Weaknesses: Slowest, most 

...

  - Hypothetical doc (first 100 chars): **Événement : "Annecy en Musique – Week-end Familial Gratuit"**
**Lieu :** Parc Charles Bosson (Anne...

[3/5] Searching with HyDE embedding...


INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"


  - Retrieved 10 documents

[4/5] Reranking documents...
  - Top 5 documents after reranking

[5/5] Generating final answer...


INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"


✓ Advanced RAG processing complete

[Execution time: 10.53s]

Query Analysis:
  Reformulated: Événements musicaux gratuits pour familles à Annecy ce week-end 2024

Retrieved Sources (5 docs):
  1. 7 juillet 2023 - HK au Festival Musical'été - Annemasse (74)
     Annemasse, Vendredi 7 juillet, 19h00
  2. Nuit théâtrale au Musée-Château
     Annecy, Samedi 18 mai, 18h00
  3. Visite famille de l’espace environnement du lac d’Annecy
     Annecy, Samedi 21 septembre 2024, 11h00, 14h00

Answer:
Voici les événements musicaux gratuits pour familles à Annecy ce week-end, d'après les informations disponibles :

1. **Nuit théâtrale au Musée-Château** (Samedi 18 mai, 18h00)
   - **Lieu** : Musée-Château d'Annecy (Place du Château, 74000 Annecy)
   - **Description** : Soirée festive avec des élèv...


SUMMARY

Approach             Time (s)     Retrieval Method                        
--------------------------------------------------------------------------------
Basic RAG            3.44         F

## 12. Evaluation Framework

This section implements a comprehensive testing and evaluation framework for all three RAG methods:
- **Test Dataset**: CSV file with manually annotated questions and expected answers
- **LLM-as-Judge**: Automated evaluation using Mistral with Chain-of-Thought reasoning
- **Statistics**: Automated calculation of PASS/PARTIAL/FAIL rates for each RAG method

In [ ]:
# Load test dataset from CSV
TEST_DATA_PATH = Path("../tests/test_data/test_questions.csv")

print("Loading test dataset...")
test_df = pd.read_csv(TEST_DATA_PATH)

print(f"✓ Test dataset loaded: {len(test_df)} questions")
print("\n" + "="*60)
print("TEST DATASET STATISTICS")
print("="*60)

# Category distribution
print("\nCategory Distribution:")
print(test_df['category'].value_counts())

# Difficulty distribution
print("\nDifficulty Distribution:")
print(test_df['difficulty'].value_counts())

# Display sample questions
print("\n" + "="*60)
print("SAMPLE TEST QUESTIONS")
print("="*60)

for idx in [0, 10, 13]:  # Show factual, complex, and off-topic examples
    row = test_df.iloc[idx]
    print(f"\n[{row['category'].upper()}] Question {row['id']}:")
    print(f"  Q: {row['question']}")
    print(f"  Expected: {row['expected_answer'][:100]}...")
    print(f"  Difficulty: {row['difficulty']}")

print("\n" + "="*60)
print(f"✓ Ready to evaluate {len(test_df)} test questions")
print("="*60)

In [ ]:
# Evaluation helper functions

def run_all_rag_methods(query: str) -> dict:
    """
    Run a query through all three RAG methods and return results.
    
    Returns:
        dict with keys: 'basic', 'hybrid', 'advanced'
        Each value is a dict with 'answer' and 'source_documents'
    """
    results = {}
    
    # Basic RAG
    try:
        basic_result = qa_chain.invoke({"query": query})
        results['basic'] = {
            'answer': basic_result['result'],
            'source_documents': basic_result['source_documents']
        }
    except Exception as e:
        results['basic'] = {
            'answer': f"[ERROR] {str(e)}",
            'source_documents': []
        }
    
    # Hybrid RAG
    try:
        hybrid_result = hybrid_qa_chain.invoke({"query": query})
        results['hybrid'] = {
            'answer': hybrid_result['result'],
            'source_documents': hybrid_result['source_documents']
        }
    except Exception as e:
        results['hybrid'] = {
            'answer': f"[ERROR] {str(e)}",
            'source_documents': []
        }
    
    # Advanced RAG
    try:
        advanced_result = advanced_rag(query)
        results['advanced'] = {
            'answer': advanced_result['answer'],
            'source_documents': advanced_result['source_documents'],
            'bypassed_search': advanced_result.get('bypassed_search', False)
        }
    except Exception as e:
        results['advanced'] = {
            'answer': f"[ERROR] {str(e)}",
            'source_documents': [],
            'bypassed_search': False
        }
    
    return results


def format_results_for_judge(question: str, expected: str, generated: str) -> dict:
    """
    Format question, expected answer, and generated answer for LLM judge evaluation.
    
    Returns:
        dict with formatted inputs for the judge
    """
    return {
        'question': question.strip(),
        'expected_answer': expected.strip(),
        'generated_answer': generated.strip()
    }


def parse_judge_verdict(judge_response: str) -> dict:
    """
    Parse the judge's response to extract verdict and explanation.
    
    Expected format:
        RAISONNEMENT: [analysis]
        VERDICT: [PASS|PARTIAL|FAIL]
        EXPLICATION: [summary]
    
    Returns:
        dict with 'verdict', 'reasoning', 'explanation'
    """
    lines = judge_response.strip().split('\n')
    
    result = {
        'verdict': 'UNKNOWN',
        'reasoning': '',
        'explanation': ''
    }
    
    current_section = None
    
    for line in lines:
        line = line.strip()
        
        if line.startswith('RAISONNEMENT:'):
            current_section = 'reasoning'
            result['reasoning'] = line.replace('RAISONNEMENT:', '').strip()
        elif line.startswith('VERDICT:'):
            current_section = 'verdict'
            verdict_text = line.replace('VERDICT:', '').strip().upper()
            # Extract PASS, PARTIAL, or FAIL
            if 'PASS' in verdict_text and 'PARTIAL' not in verdict_text:
                result['verdict'] = 'PASS'
            elif 'PARTIAL' in verdict_text:
                result['verdict'] = 'PARTIAL'
            elif 'FAIL' in verdict_text:
                result['verdict'] = 'FAIL'
        elif line.startswith('EXPLICATION:'):
            current_section = 'explanation'
            result['explanation'] = line.replace('EXPLICATION:', '').strip()
        elif current_section and line:
            # Continue accumulating content for current section
            result[current_section] += ' ' + line
    
    # Fallback: try to find verdict anywhere in response
    if result['verdict'] == 'UNKNOWN':
        if 'PASS' in judge_response.upper() and 'PARTIAL' not in judge_response.upper():
            result['verdict'] = 'PASS'
        elif 'PARTIAL' in judge_response.upper():
            result['verdict'] = 'PARTIAL'
        elif 'FAIL' in judge_response.upper():
            result['verdict'] = 'FAIL'
    
    return result


print("✓ Evaluation helper functions defined:")
print("  - run_all_rag_methods(query)")
print("  - format_results_for_judge(question, expected, generated)")
print("  - parse_judge_verdict(judge_response)")

In [ ]:
# LLM-as-Judge Implementation

# Initialize judge LLM (using Mistral Large for better reasoning)
judge_llm = ChatMistralAI(
    model="mistral-large-latest",
    api_key=MISTRAL_API_KEY,
    temperature=0  # Deterministic evaluation
)

# Judge prompt template with Chain-of-Thought reasoning
JUDGE_PROMPT_TEMPLATE = """Tu es un évaluateur expert pour un système RAG d'événements culturels.

Question posée: {question}
Réponse attendue: {expected_answer}
Réponse générée: {generated_answer}

Évalue la réponse générée selon ces critères:

1. EXACTITUDE: Les informations (dates, lieux, noms) sont-elles correctes?
2. PERTINENCE: La réponse répond-elle à la question posée?
3. COMPLÉTUDE: Les éléments essentiels sont-ils présents?

IMPORTANT - Critères d'évaluation:
- **PASS**: La réponse contient des événements corrects et pertinents. Les informations principales (dates, lieux, types d'événements) sont exactes. Peut inclure des événements supplémentaires valides non mentionnés dans la réponse attendue.
- **PARTIAL**: Certains événements sont corrects mais des éléments clés sont manquants, OU contient des inexactitudes mineures dans les détails, OU la réponse est incomplète mais pas fausse.
- **FAIL**: Les événements sont incorrects/inventés, mauvaises localisations/dates, réponse complètement hors sujet, ou aucune information utile fournie.

NOTE CRITIQUE: Une réponse avec moins d'événements que prévu mais tous corrects est PASS, pas PARTIAL.
Une réponse avec plus d'événements (tous valides) est aussi PASS.

Pour les questions OFF_TOPIC:
- Si le système détecte correctement que la question est hors sujet et refuse de chercher → PASS
- Si le système retourne des résultats non pertinents → FAIL
- Si le système dit "pas d'information" de manière appropriée → PARTIAL

Pour les questions VAGUE:
- Si le système demande des clarifications → PASS
- Si le système reformule intelligemment et trouve des résultats pertinents → PASS
- Si le système retourne des résultats aléatoires → FAIL

Raisonne étape par étape, puis donne ton verdict final.

Format de réponse STRICTEMENT OBLIGATOIRE:
RAISONNEMENT: [ton analyse détaillée en 2-3 phrases]
VERDICT: [PASS|PARTIAL|FAIL]
EXPLICATION: [résumé en une phrase]"""


def evaluate_with_judge(question: str, expected_answer: str, generated_answer: str, category: str = "factual") -> dict:
    """
    Evaluate a generated answer using LLM-as-Judge with Chain-of-Thought reasoning.
    
    Args:
        question: The question asked
        expected_answer: The expected/ground truth answer
        generated_answer: The answer generated by the RAG system
        category: Question category (factual, complex, off_topic, vague)
    
    Returns:
        dict with 'verdict', 'reasoning', 'explanation', 'raw_response'
    """
    # Format inputs
    judge_input = format_results_for_judge(question, expected_answer, generated_answer)
    
    # Create prompt
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        question=judge_input['question'],
        expected_answer=judge_input['expected_answer'],
        generated_answer=judge_input['generated_answer']
    )
    
    # Get judge evaluation
    try:
        response = judge_llm.invoke(prompt)
        raw_response = response.content if hasattr(response, 'content') else str(response)
        
        # Parse verdict
        parsed = parse_judge_verdict(raw_response)
        parsed['raw_response'] = raw_response
        parsed['category'] = category
        
        return parsed
    
    except Exception as e:
        return {
            'verdict': 'ERROR',
            'reasoning': f"Evaluation error: {str(e)}",
            'explanation': "Could not evaluate",
            'raw_response': "",
            'category': category
        }


# Test the judge with a sample evaluation
print("="*60)
print("TESTING LLM-AS-JUDGE")
print("="*60)

test_question = "Quels concerts ont lieu à Annecy?"
test_expected = "Concerts au château (Musée-Château, mai), événements musicaux au Conservatoire (septembre)"
test_generated_good = "À Annecy, il y a des concerts au château en mai et des événements musicaux au Conservatoire en septembre."
test_generated_partial = "Il y a des concerts au château à Annecy."
test_generated_bad = "Il y a un festival de jazz à Chambéry en juillet."

print("\n[TEST 1: GOOD ANSWER - Should be PASS]")
print(f"Question: {test_question}")
print(f"Generated: {test_generated_good}")

result_good = evaluate_with_judge(test_question, test_expected, test_generated_good)
print(f"\nVerdict: {result_good['verdict']}")
print(f"Reasoning: {result_good['reasoning']}")

print("\n" + "-"*60)
print("\n[TEST 2: PARTIAL ANSWER - Should be PARTIAL]")
print(f"Question: {test_question}")
print(f"Generated: {test_generated_partial}")

result_partial = evaluate_with_judge(test_question, test_expected, test_generated_partial)
print(f"\nVerdict: {result_partial['verdict']}")
print(f"Reasoning: {result_partial['reasoning']}")

print("\n" + "-"*60)
print("\n[TEST 3: BAD ANSWER - Should be FAIL]")
print(f"Question: {test_question}")
print(f"Generated: {test_generated_bad}")

result_bad = evaluate_with_judge(test_question, test_expected, test_generated_bad)
print(f"\nVerdict: {result_bad['verdict']}")
print(f"Reasoning: {result_bad['reasoning']}")

print("\n" + "="*60)
print("✓ LLM-as-Judge system initialized and tested")
print("="*60)

In [ ]:
# Automated Test Runner - Run all test questions through all RAG methods
# WARNING: This will make many API calls and may take 10-15 minutes
# Set RUN_FULL_EVALUATION = True to run, or False to skip

RUN_FULL_EVALUATION = False  # Set to True to run full evaluation

if RUN_FULL_EVALUATION:
    print("="*60)
    print("RUNNING FULL EVALUATION")
    print("="*60)
    print(f"Total questions: {len(test_df)}")
    print(f"RAG methods: 3 (Basic, Hybrid, Advanced)")
    print(f"Estimated time: ~{len(test_df) * 0.5:.0f} minutes")
    print("="*60)
    
    # Store all results
    all_results = []
    
    for idx, row in test_df.iterrows():
        question_id = row['id']
        question = row['question']
        expected = row['expected_answer']
        category = row['category']
        difficulty = row['difficulty']
        
        print(f"\n[{idx+1}/{len(test_df)}] Testing Question {question_id}: {category}/{difficulty}")
        print(f"  Q: {question[:60]}...")
        
        # Run all RAG methods
        try:
            rag_results = run_all_rag_methods(question)
            
            # Evaluate each method
            for method_name in ['basic', 'hybrid', 'advanced']:
                method_result = rag_results[method_name]
                generated_answer = method_result['answer']
                
                print(f"    [{method_name.upper()}] Evaluating...")
                
                # Evaluate with judge
                evaluation = evaluate_with_judge(
                    question=question,
                    expected_answer=expected,
                    generated_answer=generated_answer,
                    category=category
                )
                
                # Store result
                result_entry = {
                    'question_id': question_id,
                    'question': question,
                    'expected_answer': expected,
                    'category': category,
                    'difficulty': difficulty,
                    'method': method_name,
                    'generated_answer': generated_answer,
                    'verdict': evaluation['verdict'],
                    'reasoning': evaluation['reasoning'],
                    'explanation': evaluation['explanation'],
                    'num_sources': len(method_result.get('source_documents', []))
                }
                
                all_results.append(result_entry)
                
                print(f"      → {evaluation['verdict']}")
        
        except Exception as e:
            print(f"    ERROR: {str(e)}")
            # Record error for all methods
            for method_name in ['basic', 'hybrid', 'advanced']:
                all_results.append({
                    'question_id': question_id,
                    'question': question,
                    'expected_answer': expected,
                    'category': category,
                    'difficulty': difficulty,
                    'method': method_name,
                    'generated_answer': f"[ERROR] {str(e)}",
                    'verdict': 'ERROR',
                    'reasoning': 'Test execution error',
                    'explanation': str(e),
                    'num_sources': 0
                })
    
    # Convert to DataFrame
    results_df = pd.DataFrame(all_results)
    
    print("\n" + "="*60)
    print("✓ EVALUATION COMPLETE")
    print("="*60)
    print(f"Total evaluations: {len(results_df)}")
    print(f"Questions tested: {len(test_df)}")
    print(f"Methods tested: 3 (Basic, Hybrid, Advanced)")
    print("="*60)
    
else:
    print("="*60)
    print("FULL EVALUATION SKIPPED")
    print("="*60)
    print("To run full evaluation, set RUN_FULL_EVALUATION = True")
    print("\nCreating sample results for demonstration...")
    
    # Create sample results with first 3 questions
    sample_results = []
    
    for idx, row in test_df.head(3).iterrows():
        question_id = row['id']
        question = row['question']
        expected = row['expected_answer']
        category = row['category']
        difficulty = row['difficulty']
        
        print(f"\n[SAMPLE {idx+1}/3] Testing Question {question_id}")
        print(f"  Q: {question}")
        
        # Run all RAG methods
        rag_results = run_all_rag_methods(question)
        
        # Evaluate each method
        for method_name in ['basic', 'hybrid', 'advanced']:
            method_result = rag_results[method_name]
            generated_answer = method_result['answer']
            
            print(f"  [{method_name.upper()}] Evaluating...")
            
            # Evaluate with judge
            evaluation = evaluate_with_judge(
                question=question,
                expected_answer=expected,
                generated_answer=generated_answer,
                category=category
            )
            
            # Store result
            result_entry = {
                'question_id': question_id,
                'question': question,
                'expected_answer': expected,
                'category': category,
                'difficulty': difficulty,
                'method': method_name,
                'generated_answer': generated_answer,
                'verdict': evaluation['verdict'],
                'reasoning': evaluation['reasoning'],
                'explanation': evaluation['explanation'],
                'num_sources': len(method_result.get('source_documents', []))
            }
            
            sample_results.append(result_entry)
            
            print(f"    → {evaluation['verdict']}")
    
    # Convert to DataFrame
    results_df = pd.DataFrame(sample_results)
    
    print("\n" + "="*60)
    print("✓ SAMPLE EVALUATION COMPLETE")
    print("="*60)
    print(f"Sample evaluations: {len(results_df)}")
    print("Set RUN_FULL_EVALUATION = True to evaluate all questions")
    print("="*60)

# Display first few results
print("\nSample Results:")
print(results_df[['question_id', 'method', 'verdict', 'category']].head(9))

In [ ]:
# Statistics Generation - Calculate % PASS, % PARTIAL, % FAIL per RAG method

def calculate_statistics(results_dataframe):
    """
    Calculate evaluation statistics for all RAG methods.
    
    Returns:
        dict with statistics for each method
    """
    stats = {}
    
    for method in ['basic', 'hybrid', 'advanced']:
        method_results = results_dataframe[results_dataframe['method'] == method]
        total = len(method_results)
        
        if total == 0:
            stats[method] = {
                'total': 0,
                'pass_count': 0,
                'partial_count': 0,
                'fail_count': 0,
                'error_count': 0,
                'pass_pct': 0.0,
                'partial_pct': 0.0,
                'fail_pct': 0.0,
                'error_pct': 0.0
            }
            continue
        
        pass_count = len(method_results[method_results['verdict'] == 'PASS'])
        partial_count = len(method_results[method_results['verdict'] == 'PARTIAL'])
        fail_count = len(method_results[method_results['verdict'] == 'FAIL'])
        error_count = len(method_results[method_results['verdict'] == 'ERROR'])
        
        stats[method] = {
            'total': total,
            'pass_count': pass_count,
            'partial_count': partial_count,
            'fail_count': fail_count,
            'error_count': error_count,
            'pass_pct': (pass_count / total) * 100,
            'partial_pct': (partial_count / total) * 100,
            'fail_pct': (fail_count / total) * 100,
            'error_pct': (error_count / total) * 100
        }
    
    return stats


def print_statistics_table(stats):
    """
    Print a nicely formatted statistics table.
    """
    print("\n" + "╔" + "═"*77 + "╗")
    print("║" + " "*26 + "EVALUATION RESULTS" + " "*33 + "║")
    print("╠" + "═"*77 + "╣")
    print("║ Method        │ PASS        │ PARTIAL     │ FAIL        │ Total       ║")
    print("╠" + "═"*77 + "╣")
    
    for method_name, method_stats in stats.items():
        method_display = method_name.capitalize().ljust(13)
        total = method_stats['total']
        
        pass_str = f"{method_stats['pass_pct']:.0f}% ({method_stats['pass_count']})".ljust(11)
        partial_str = f"{method_stats['partial_pct']:.0f}% ({method_stats['partial_count']})".ljust(11)
        fail_str = f"{method_stats['fail_pct']:.0f}% ({method_stats['fail_count']})".ljust(11)
        total_str = f"{total}".ljust(11)
        
        print(f"║ {method_display} │ {pass_str} │ {partial_str} │ {fail_str} │ {total_str} ║")
    
    print("╚" + "═"*77 + "╝")


def print_category_breakdown(results_dataframe):
    """
    Print breakdown by category for each method.
    """
    print("\n" + "="*60)
    print("BREAKDOWN BY CATEGORY")
    print("="*60)
    
    categories = results_dataframe['category'].unique()
    
    for category in sorted(categories):
        print(f"\n{category.upper()}:")
        category_results = results_dataframe[results_dataframe['category'] == category]
        
        for method in ['basic', 'hybrid', 'advanced']:
            method_results = category_results[category_results['method'] == method]
            if len(method_results) == 0:
                continue
            
            total = len(method_results)
            pass_count = len(method_results[method_results['verdict'] == 'PASS'])
            partial_count = len(method_results[method_results['verdict'] == 'PARTIAL'])
            fail_count = len(method_results[method_results['verdict'] == 'FAIL'])
            
            pass_pct = (pass_count / total) * 100 if total > 0 else 0
            partial_pct = (partial_count / total) * 100 if total > 0 else 0
            fail_pct = (fail_count / total) * 100 if total > 0 else 0
            
            print(f"  {method.capitalize():10} - PASS: {pass_pct:5.0f}% ({pass_count}/{total}) | "
                  f"PARTIAL: {partial_pct:5.0f}% ({partial_count}/{total}) | "
                  f"FAIL: {fail_pct:5.0f}% ({fail_count}/{total})")


# Calculate and display statistics
print("="*60)
print("GENERATING STATISTICS")
print("="*60)

stats = calculate_statistics(results_df)

# Print main statistics table
print_statistics_table(stats)

# Print category breakdown
print_category_breakdown(results_df)

# Print difficulty breakdown (if available)
if 'difficulty' in results_df.columns:
    print("\n" + "="*60)
    print("BREAKDOWN BY DIFFICULTY")
    print("="*60)
    
    difficulties = results_df['difficulty'].unique()
    
    for difficulty in sorted(difficulties):
        print(f"\n{difficulty.upper()}:")
        diff_results = results_df[results_df['difficulty'] == difficulty]
        
        for method in ['basic', 'hybrid', 'advanced']:
            method_results = diff_results[diff_results['method'] == method]
            if len(method_results) == 0:
                continue
            
            total = len(method_results)
            pass_count = len(method_results[method_results['verdict'] == 'PASS'])
            partial_count = len(method_results[method_results['verdict'] == 'PARTIAL'])
            fail_count = len(method_results[method_results['verdict'] == 'FAIL'])
            
            pass_pct = (pass_count / total) * 100 if total > 0 else 0
            partial_pct = (partial_count / total) * 100 if total > 0 else 0
            fail_pct = (fail_count / total) * 100 if total > 0 else 0
            
            print(f"  {method.capitalize():10} - PASS: {pass_pct:5.0f}% ({pass_count}/{total}) | "
                  f"PARTIAL: {partial_pct:5.0f}% ({partial_count}/{total}) | "
                  f"FAIL: {fail_pct:5.0f}% ({fail_count}/{total})")

print("\n" + "="*60)
print("✓ Statistics generation complete")
print("="*60)

In [ ]:
# RAGAS Metrics (Optional/Advanced)
# This cell demonstrates how to use RAGAS for automated evaluation metrics
# Requires: pip install ragas

USE_RAGAS = False  # Set to True to use RAGAS metrics

if USE_RAGAS:
    try:
        from ragas import evaluate
        from ragas.metrics import (
            faithfulness,
            answer_relevancy,
            context_precision,
            context_recall
        )
        from datasets import Dataset
        
        print("="*60)
        print("RAGAS EVALUATION")
        print("="*60)
        
        # Prepare data for RAGAS
        # RAGAS expects: question, answer, contexts, ground_truth
        ragas_data = {
            'question': [],
            'answer': [],
            'contexts': [],
            'ground_truth': []
        }
        
        # Use only basic RAG results for RAGAS demo
        basic_results = results_df[results_df['method'] == 'basic']
        
        for idx, row in basic_results.iterrows():
            ragas_data['question'].append(row['question'])
            ragas_data['answer'].append(row['generated_answer'])
            ragas_data['ground_truth'].append(row['expected_answer'])
            # Note: We'd need to store contexts from retrieval for this
            # For now, using empty list as placeholder
            ragas_data['contexts'].append([])
        
        # Create dataset
        dataset = Dataset.from_dict(ragas_data)
        
        # Run RAGAS evaluation
        print("\nRunning RAGAS evaluation (this may take a few minutes)...")
        result = evaluate(
            dataset,
            metrics=[
                faithfulness,
                answer_relevancy,
                context_precision,
                context_recall
            ]
        )
        
        print("\n" + "="*60)
        print("RAGAS METRICS RESULTS")
        print("="*60)
        print(result)
        
        print("\n✓ RAGAS evaluation complete")
        
    except ImportError:
        print("="*60)
        print("RAGAS NOT INSTALLED")
        print("="*60)
        print("To use RAGAS metrics, install with:")
        print("  pip install ragas")
        print("\nRAGAS provides automated metrics:")
        print("  - Faithfulness: How factual is the generated answer")
        print("  - Answer Relevancy: How relevant is the answer to the question")
        print("  - Context Precision: How precise is the retrieved context")
        print("  - Context Recall: How complete is the retrieved context")
        print("="*60)
        
    except Exception as e:
        print(f"Error running RAGAS: {e}")
        print("Continuing without RAGAS metrics...")

else:
    print("="*60)
    print("RAGAS EVALUATION SKIPPED (OPTIONAL)")
    print("="*60)
    print("RAGAS is an optional advanced evaluation framework.")
    print("\nTo enable RAGAS metrics:")
    print("  1. Install: pip install ragas")
    print("  2. Set USE_RAGAS = True in this cell")
    print("\nRAGAS provides automated metrics:")
    print("  - Faithfulness: Factual consistency with retrieved context")
    print("  - Answer Relevancy: Relevance of answer to question")
    print("  - Context Precision: Precision of retrieved documents")
    print("  - Context Recall: Completeness of retrieved information")
    print("\nNote: RAGAS requires storing retrieval contexts, which would")
    print("need to be added to the test runner to work properly.")
    print("="*60)

In [ ]:
# Results Export - Save detailed results and summary report

import json
from datetime import datetime

# Create output directory
OUTPUT_DIR = Path("../evaluation_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# Generate timestamp for filenames
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. Export detailed results to CSV
csv_filename = OUTPUT_DIR / f"evaluation_results_{timestamp}.csv"
results_df.to_csv(csv_filename, index=False, encoding='utf-8')
print(f"✓ Detailed results exported to CSV: {csv_filename}")

# 2. Export detailed results to JSON
json_filename = OUTPUT_DIR / f"evaluation_results_{timestamp}.json"
results_json = results_df.to_dict(orient='records')
with open(json_filename, 'w', encoding='utf-8') as f:
    json.dump(results_json, f, ensure_ascii=False, indent=2)
print(f"✓ Detailed results exported to JSON: {json_filename}")

# 3. Generate and export summary report
summary_filename = OUTPUT_DIR / f"evaluation_summary_{timestamp}.txt"

with open(summary_filename, 'w', encoding='utf-8') as f:
    # Header
    f.write("="*70 + "\n")
    f.write("RAG EVALUATION SUMMARY REPORT\n")
    f.write("="*70 + "\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Total Questions: {len(test_df)}\n")
    f.write(f"Questions Evaluated: {len(results_df) // 3}\n")  # Divide by 3 methods
    f.write(f"RAG Methods: Basic, Hybrid, Advanced\n")
    f.write("="*70 + "\n\n")
    
    # Statistics table
    stats = calculate_statistics(results_df)
    
    f.write("OVERALL RESULTS\n")
    f.write("="*70 + "\n")
    f.write(f"{'Method':<15} {'PASS':<15} {'PARTIAL':<15} {'FAIL':<15} {'Total':<10}\n")
    f.write("-"*70 + "\n")
    
    for method_name, method_stats in stats.items():
        f.write(f"{method_name.capitalize():<15} "
                f"{method_stats['pass_pct']:.0f}% ({method_stats['pass_count']})"
                f"{'':<7} "
                f"{method_stats['partial_pct']:.0f}% ({method_stats['partial_count']})"
                f"{'':<7} "
                f"{method_stats['fail_pct']:.0f}% ({method_stats['fail_count']})"
                f"{'':<7} "
                f"{method_stats['total']}\n")
    
    f.write("="*70 + "\n\n")
    
    # Category breakdown
    f.write("BREAKDOWN BY CATEGORY\n")
    f.write("="*70 + "\n")
    
    categories = results_df['category'].unique()
    for category in sorted(categories):
        f.write(f"\n{category.upper()}:\n")
        category_results = results_df[results_df['category'] == category]
        
        for method in ['basic', 'hybrid', 'advanced']:
            method_results = category_results[category_results['method'] == method]
            if len(method_results) == 0:
                continue
            
            total = len(method_results)
            pass_count = len(method_results[method_results['verdict'] == 'PASS'])
            partial_count = len(method_results[method_results['verdict'] == 'PARTIAL'])
            fail_count = len(method_results[method_results['verdict'] == 'FAIL'])
            
            pass_pct = (pass_count / total) * 100 if total > 0 else 0
            partial_pct = (partial_count / total) * 100 if total > 0 else 0
            fail_pct = (fail_count / total) * 100 if total > 0 else 0
            
            f.write(f"  {method.capitalize():<12} - "
                    f"PASS: {pass_pct:5.0f}% ({pass_count}/{total}) | "
                    f"PARTIAL: {partial_pct:5.0f}% ({partial_count}/{total}) | "
                    f"FAIL: {fail_pct:5.0f}% ({fail_count}/{total})\n")
    
    f.write("\n" + "="*70 + "\n\n")
    
    # Key findings
    f.write("KEY FINDINGS\n")
    f.write("="*70 + "\n\n")
    
    # Find best performing method
    best_method = max(stats.items(), key=lambda x: x[1]['pass_pct'])
    f.write(f"Best Overall Performance: {best_method[0].capitalize()} "
            f"({best_method[1]['pass_pct']:.0f}% PASS rate)\n\n")
    
    # Method comparisons
    f.write("Method Comparison:\n")
    f.write(f"- Basic RAG: {stats['basic']['pass_pct']:.0f}% PASS, "
            f"{stats['basic']['partial_pct']:.0f}% PARTIAL, "
            f"{stats['basic']['fail_pct']:.0f}% FAIL\n")
    f.write(f"- Hybrid RAG: {stats['hybrid']['pass_pct']:.0f}% PASS, "
            f"{stats['hybrid']['partial_pct']:.0f}% PARTIAL, "
            f"{stats['hybrid']['fail_pct']:.0f}% FAIL\n")
    f.write(f"- Advanced RAG: {stats['advanced']['pass_pct']:.0f}% PASS, "
            f"{stats['advanced']['partial_pct']:.0f}% PARTIAL, "
            f"{stats['advanced']['fail_pct']:.0f}% FAIL\n")
    
    f.write("\n" + "="*70 + "\n")
    f.write("END OF REPORT\n")
    f.write("="*70 + "\n")

print(f"✓ Summary report exported: {summary_filename}")

# 4. Display export summary
print("\n" + "="*70)
print("EXPORT SUMMARY")
print("="*70)
print(f"\nExported Files:")
print(f"  1. Detailed CSV:  {csv_filename.name}")
print(f"  2. Detailed JSON: {json_filename.name}")
print(f"  3. Summary TXT:   {summary_filename.name}")
print(f"\nOutput Directory: {OUTPUT_DIR.absolute()}")
print(f"\nTotal Records: {len(results_df)}")
print(f"File Sizes:")
print(f"  - CSV:  {csv_filename.stat().st_size / 1024:.1f} KB")
print(f"  - JSON: {json_filename.stat().st_size / 1024:.1f} KB")
print(f"  - TXT:  {summary_filename.stat().st_size / 1024:.1f} KB")

print("\n" + "="*70)
print("✓ All evaluation results exported successfully")
print("="*70)

print("\nNext Steps:")
print("  1. Review detailed results in CSV/JSON files")
print("  2. Include summary report in technical documentation")
print("  3. Analyze failures to improve RAG system")
print("  4. Expand test dataset for more comprehensive evaluation")